# Modelo logistico 2026

Este notebook entrena y evalua una regresion logistica para clasificar si un evento armado de 2026 tiene fatalidades reportadas.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

BASE_DIR = Path('..').resolve()
sys.path.append(str(BASE_DIR))

from scripts.train_logreg_2026 import (
    TARGET_COLUMN,
    coefficients_frame,
    evaluate_split,
    feature_columns,
    load_2026_data,
    make_pipeline,
    split_random,
    split_temporal,
)

df = load_2026_data()
df.shape

## Balance de clases

In [ ]:
df[TARGET_COLUMN].value_counts(normalize=False).rename(index={0: 'no_letal', 1: 'letal'}), df[TARGET_COLUMN].mean()

## Splits

La validacion temporal es la referencia principal. El split aleatorio estratificado se usa como diagnostico de senal interna.

In [ ]:
feature_cols, numeric_cols, categorical_cols = feature_columns(df)
temporal_train, temporal_test = split_temporal(df)
random_train, random_test = split_random(df)

split_summary = pd.DataFrame(
    [
        {'split': 'temporal_train', 'rows': len(temporal_train), 'positive_rate': temporal_train[TARGET_COLUMN].mean()},
        {'split': 'temporal_test_may', 'rows': len(temporal_test), 'positive_rate': temporal_test[TARGET_COLUMN].mean()},
        {'split': 'random_train', 'rows': len(random_train), 'positive_rate': random_train[TARGET_COLUMN].mean()},
        {'split': 'random_test', 'rows': len(random_test), 'positive_rate': random_test[TARGET_COLUMN].mean()},
    ]
)
split_summary

## Entrenamiento

In [ ]:
variants = [
    {'name': 'logreg_l2_balanced', 'feature_profile': 'full', 'class_weight': 'balanced', 'c_value': 1.0, 'l1_ratio': 0.0},
    {'name': 'logreg_l2_unweighted', 'feature_profile': 'full', 'class_weight': None, 'c_value': 1.0, 'l1_ratio': 0.0},
    {'name': 'logreg_l1_balanced_c1', 'feature_profile': 'full', 'class_weight': 'balanced', 'c_value': 1.0, 'l1_ratio': 1.0},
    {'name': 'logreg_l1_unweighted_c1', 'feature_profile': 'full', 'class_weight': None, 'c_value': 1.0, 'l1_ratio': 1.0},
    {'name': 'logreg_l1_balanced_c03', 'feature_profile': 'full', 'class_weight': 'balanced', 'c_value': 0.3, 'l1_ratio': 1.0},
    {'name': 'logreg_l1_unweighted_c03', 'feature_profile': 'full', 'class_weight': None, 'c_value': 0.3, 'l1_ratio': 1.0},
    {'name': 'logreg_l1_balanced_c01', 'feature_profile': 'full', 'class_weight': 'balanced', 'c_value': 0.1, 'l1_ratio': 1.0},
    {'name': 'logreg_l1_unweighted_c01', 'feature_profile': 'full', 'class_weight': None, 'c_value': 0.1, 'l1_ratio': 1.0},
    {'name': 'logreg_core_l1_balanced_c01', 'feature_profile': 'core_interpretable', 'class_weight': 'balanced', 'c_value': 0.1, 'l1_ratio': 1.0},
    {'name': 'logreg_core_l1_unweighted_c01', 'feature_profile': 'core_interpretable', 'class_weight': None, 'c_value': 0.1, 'l1_ratio': 1.0},
]

metrics = []
predictions = []
models = {}
model_configs = {}
model_feature_specs = {}

for config in variants:
    variant = config['name']
    feature_profile = config['feature_profile']
    feature_cols, numeric_cols, categorical_cols = feature_columns(df, feature_profile)
    temporal_model = make_pipeline(
        numeric_cols,
        categorical_cols,
        config['class_weight'],
        config['c_value'],
        config['l1_ratio'],
    )
    temporal_metrics, temporal_predictions = evaluate_split(
        'temporal_train_until_apr_test_may',
        variant,
        feature_profile,
        temporal_model,
        temporal_train,
        temporal_test,
        feature_cols,
    )
    temporal_metrics['c_value'] = config['c_value']
    temporal_metrics['l1_ratio'] = config['l1_ratio']
    temporal_metrics['class_weight'] = config['class_weight'] or 'none'
    metrics.append(temporal_metrics)
    predictions.append(temporal_predictions)
    models[variant] = temporal_model
    model_configs[variant] = config
    model_feature_specs[variant] = (numeric_cols, categorical_cols)

    random_model = make_pipeline(
        numeric_cols,
        categorical_cols,
        config['class_weight'],
        config['c_value'],
        config['l1_ratio'],
    )
    random_metrics, random_predictions = evaluate_split(
        'random_stratified_80_20',
        variant,
        feature_profile,
        random_model,
        random_train,
        random_test,
        feature_cols,
    )
    random_metrics['c_value'] = config['c_value']
    random_metrics['l1_ratio'] = config['l1_ratio']
    random_metrics['class_weight'] = config['class_weight'] or 'none'
    metrics.append(random_metrics)
    predictions.append(random_predictions)

metrics_df = pd.DataFrame(metrics)
predictions_df = pd.concat(predictions, ignore_index=True)

metrics_df

## Matriz de confusion

In [ ]:
metrics_df[[
    'split', 'variant', 'c_value', 'l1_ratio', 'class_weight',
    'tn', 'fp', 'fn', 'tp', 'precision', 'recall', 'f1',
    'roc_auc', 'average_precision', 'brier_score',
    'nonzero_features', 'zeroed_features'
]]

## Coeficientes interpretables

Los coeficientes se calculan sobre el modelo temporal, que es la validacion principal.

In [ ]:
coefs = pd.concat(
    [
        coefficients_frame(
            variant,
            model,
            model_feature_specs[variant][0],
            model_feature_specs[variant][1],
            model_configs[variant],
        )
        for variant, model in models.items()
    ],
    ignore_index=True,
)

coefs.sort_values('coefficient', ascending=False).head(20)

In [ ]:
coefs.sort_values('coefficient').head(20)

## Seleccion L1

La regularizacion L1 deja coeficientes exactamente en cero. Para explicabilidad, miramos cuantas features sobreviven y revisamos el modelo `logreg_l1_balanced_c01` como candidato compacto.

In [ ]:
selected_summary = (
    coefs[coefs['selected_l1']]
    .groupby('variant')
    .agg(selected_features=('feature', 'size'), max_abs_coefficient=('abs_coefficient', 'max'))
    .sort_values('selected_features')
)
selected_summary

In [ ]:
candidate = 'logreg_l1_balanced_c01'
coefs[(coefs['variant'].eq(candidate)) & (coefs['selected_l1'])][
    ['feature', 'coefficient', 'odds_ratio']
].sort_values('coefficient', ascending=False)

## Auditoria de errores

Los falsos positivos y falsos negativos de mayo son la forma mas util de entender que esta aprendiendo el modelo.

In [ ]:
temporal_predictions = predictions_df[predictions_df['split'].eq('temporal_train_until_apr_test_may')].copy()

false_positives = temporal_predictions[
    temporal_predictions['fatality_positive'].eq(0) & temporal_predictions['prediction_0_5'].eq(1)
].sort_values('fatality_probability', ascending=False)

false_negatives = temporal_predictions[
    temporal_predictions['fatality_positive'].eq(1) & temporal_predictions['prediction_0_5'].eq(0)
].sort_values('fatality_probability')

false_positives.head(10), false_negatives.head(10)

## Persistencia

Para regenerar las salidas oficiales del proyecto, ejecutar desde la raiz:

```bash
python scripts/train_logreg_2026.py
```